# ЛР-1: PySpark batch processing

Среда: **Google Colab**.

Полное методическое описание, критерии оценивания и `TEACH CARD` находятся в одноимённом `.md` файле комплекта.

Сквозной пайплайн курса:

$$
\text{Sense} \rightarrow \text{Collect} \rightarrow \text{Stream} \rightarrow
\text{Store} \rightarrow \text{Process} \rightarrow \text{Learn} \rightarrow \text{Teach}
$$


In [ ]:
!pip -q install "pyspark==4.2.0" pandas pyarrow

from pyspark.sql import SparkSession, functions as F
from pyspark.storagelevel import StorageLevel
import time

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("Lab01_RobotBatchProcessing")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

print("Spark:", spark.version)
print("Default parallelism:", spark.sparkContext.defaultParallelism)

# 1. Генерация большого журнала телеметрии без внешнего датасета.
N = 500_000

logs = (
    spark.range(N)
    .withColumn("robot_id", F.concat(F.lit("robot-"), (F.col("id") % 8).cast("string")))
    .withColumn("sensor_id", F.concat(F.lit("imu-"), (F.col("id") % 4).cast("string")))
    .withColumn("ts_ms", (F.lit(1_720_000_000_000) + F.col("id") * 50).cast("long"))
    .withColumn("ax", F.sin(F.col("id") / 25.0) + ((F.col("id") % 7) - 3) * 0.01)
    .withColumn("ay", F.cos(F.col("id") / 40.0) + ((F.col("id") % 5) - 2) * 0.01)
    .withColumn("az", F.lit(9.81) + F.sin(F.col("id") / 60.0) * 0.08)
    .withColumn("motor_temp", F.lit(42.0) + (F.col("id") % 100) * 0.12)
    .withColumn("battery_v", F.lit(12.6) - (F.col("id") % 1000) * 0.0008)
    .drop("id")
    .repartition(8, "robot_id")
)

print("Partitions:", logs.rdd.getNumPartitions())
logs.show(5, truncate=False)

# 2. Кэширование набора: повторные аналитические запросы не должны
#    заново строить весь lineage.
logs.persist(StorageLevel.MEMORY_AND_DISK)
_ = logs.count()

# 3. MapReduce-подобная агрегация: key = robot_id.
t0 = time.perf_counter()

summary = (
    logs.groupBy("robot_id")
    .agg(
        F.count("*").alias("records"),
        F.avg("motor_temp").alias("avg_motor_temp"),
        F.max("motor_temp").alias("max_motor_temp"),
        F.avg("battery_v").alias("avg_battery_v"),
        F.sqrt(F.avg(F.pow("ax", 2) + F.pow("ay", 2) + F.pow("az", 2))).alias("accel_rms")
    )
    .orderBy("robot_id")
)

summary.show(truncate=False)
elapsed = time.perf_counter() - t0
print(f"Aggregation time: {elapsed:.3f} s")

# 4. Spark SQL над тем же DataFrame.
logs.createOrReplaceTempView("robot_logs")

sql_result = spark.sql("""
SELECT
    robot_id,
    COUNT(*) AS n,
    ROUND(AVG(motor_temp), 3) AS avg_temp,
    ROUND(MAX(motor_temp), 3) AS max_temp,
    ROUND(AVG(battery_v), 4) AS avg_voltage
FROM robot_logs
GROUP BY robot_id
HAVING MAX(motor_temp) > 50
ORDER BY max_temp DESC
""")

sql_result.show(truncate=False)

# 5. План выполнения: найти Exchange/HashAggregate и обсудить shuffle.
summary.explain(mode="formatted")

# 6. Проверка распределения записей по partition.
partition_sizes = (
    logs.rdd
    .mapPartitionsWithIndex(lambda idx, it: [(idx, sum(1 for _ in it))])
    .collect()
)
print("Partition sizes:", partition_sizes)

assert sum(x[1] for x in partition_sizes) == N
assert summary.count() == 8

logs.unpersist()


## TEACH CARD

После выполнения кода заполните `TEACH CARD` из `.md`-файла лабораторной работы и приложите его к отчёту.
